In [1]:
!pip install pandas matplotlib seaborn numpy

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

print("All libraries loaded!")

All libraries loaded!


In [3]:
matches    = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

print("Matches shape   :", matches.shape)
print("Deliveries shape:", deliveries.shape)

Matches shape   : (1095, 20)
Deliveries shape: (260920, 17)


In [4]:
matches.head()            # see first 5 rows
matches.columns.tolist()  # see all column names
matches.info()            # data types + nulls
matches.describe()        # numerical summary
matches['season'].unique()         # all seasons
matches['toss_decision'].value_counts()  # bat vs field counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1095 non-null   int64  
 1   season           1095 non-null   object 
 2   city             1044 non-null   object 
 3   date             1095 non-null   object 
 4   match_type       1095 non-null   object 
 5   player_of_match  1090 non-null   object 
 6   venue            1095 non-null   object 
 7   team1            1095 non-null   object 
 8   team2            1095 non-null   object 
 9   toss_winner      1095 non-null   object 
 10  toss_decision    1095 non-null   object 
 11  winner           1090 non-null   object 
 12  result           1095 non-null   object 
 13  result_margin    1076 non-null   float64
 14  target_runs      1092 non-null   float64
 15  target_overs     1092 non-null   float64
 16  super_over       1095 non-null   object 
 17  method        

toss_decision
field    704
bat      391
Name: count, dtype: int64

In [5]:
print(matches.isnull().sum())
print(deliveries.isnull().sum())

id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64


In [6]:
matches = matches.dropna(subset=['winner'])

In [7]:
team_rename = {
    "Delhi Daredevils":       "Delhi Capitals",
    "Kings XI Punjab":        "Punjab Kings",
    "Deccan Chargers":        "Sunrisers Hyderabad",
    "Rising Pune Supergiant": "Lucknow Supergiants",
}
for col in ["team1","team2","toss_winner","winner"]:
    matches[col] = matches[col].replace(team_rename)

In [8]:
season_map = {"2007/08":"2008","2009/10":"2010","2020/21":"2021"}
matches["season"] = matches["season"].replace(season_map)
matches["season"] = matches["season"].astype(int)

In [9]:
# Did toss winner also win the match?
matches["toss_won_match"] = (
    matches["toss_winner"] == matches["winner"]
)
matches["toss_won_match"].value_counts()

toss_won_match
True     554
False    536
Name: count, dtype: int64

In [10]:
full = deliveries.merge(
    matches[["id","season","venue","toss_winner",
             "toss_decision","winner"]],
    left_on="match_id", right_on="id"
)
print("Merged shape:", full.shape)

Merged shape: (260430, 23)


In [11]:
total = len(matches)
toss_wins = matches["toss_won_match"].sum()
print(f"Toss winner won: {toss_wins/total*100:.1f}%")

Toss winner won: 50.8%


In [12]:
dec = matches.groupby("toss_decision").apply(
    lambda x: x["toss_won_match"].sum()/len(x)*100
).reset_index()
dec.columns = ["decision","win_pct"]
print(dec)

  decision    win_pct
0      bat  45.384615
1    field  53.857143


In [13]:
season_pct = matches.groupby("season").apply(
    lambda x: x["toss_won_match"].sum()/len(x)*100
).reset_index()
season_pct.columns = ["season","win_pct"]

In [14]:
venue_pct = matches.groupby("venue").apply(
    lambda x: pd.Series({
        "win_pct": x["toss_won_match"].sum()/len(x)*100,
        "matches": len(x)
    })
).reset_index()
venue_pct = venue_pct[venue_pct["matches"]>=30]
venue_pct = venue_pct.sort_values("win_pct",ascending=False)
print(venue_pct)

                                         venue    win_pct  matches
14                                Eden Gardens  55.844156     77.0
23                       M Chinnaswamy Stadium  55.555556     63.0
56                    Wankhede Stadium, Mumbai  55.555556     45.0
46                      Sawai Mansingh Stadium  53.191489     47.0
16                            Feroz Shah Kotla  52.542373     59.0
27             MA Chidambaram Stadium, Chepauk  52.083333     48.0
55                            Wankhede Stadium  50.684932     73.0
40  Punjab Cricket Association Stadium, Mohali  45.714286     35.0
13         Dubai International Cricket Stadium  39.130435     46.0
42   Rajiv Gandhi International Stadium, Uppal  34.693878     49.0


In [15]:
team_pct = matches.groupby("toss_winner").apply(
    lambda x: x["toss_won_match"].sum()/len(x)*100
).reset_index()
team_pct.columns = ["team","win_pct"]
team_pct = team_pct.sort_values("win_pct",ascending=False)
print(team_pct)

                           team    win_pct
7           Lucknow Supergiants  83.333333
2                 Gujarat Lions  66.666667
3                Gujarat Titans  63.636364
0           Chennai Super Kings  61.983471
5         Kolkata Knight Riders  55.737705
8                Mumbai Indians  54.545455
6          Lucknow Super Giants  52.631579
13  Royal Challengers Bangalore  50.892857
11             Rajasthan Royals  50.847458
4          Kochi Tuskers Kerala  50.000000
14  Royal Challengers Bengaluru  50.000000
1                Delhi Capitals  47.286822
15          Sunrisers Hyderabad  43.511450
12      Rising Pune Supergiants  42.857143
10                 Punjab Kings  41.284404
9                 Pune Warriors  15.000000


In [18]:
plt.savefig("chart/ipl_analysis.png", 
            dpi=180, bbox_inches="tight")
plt.show()

<Figure size 640x480 with 0 Axes>